# 提交说明（Chapter 4）

本 notebook 对应“标准化-映射-门控”机制实现。

- 请在运行前配置下一个代码单元中的路径参数。
- 提交到 GitHub 前请清理输出并移除敏感路径信息。
- 建议保留固定执行顺序，保证结果可复核。

In [ ]:
# 参数区（提交版本）
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("CH5_DATA_ROOT", "./data_sample"))
OUTPUT_ROOT = Path(os.environ.get("CH5_OUTPUT_ROOT", "./outputs"))
MAPPING_ROOT = Path(os.environ.get("CH5_MAPPING_ROOT", "./mapping"))

for p in [DATA_ROOT, OUTPUT_ROOT, MAPPING_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MAPPING_ROOT:", MAPPING_ROOT)

## 客户处理目录

本 notebook 按客户分节组织。建议阅读顺序：
1. 参数区与函数定义
2. 各客户处理分节（国大、竹林、一心堂等）
3. 全量整合与日期范围生成

### Offtake Pre-Processing for Core Brands BU
#### by Eric Wu, Feb 2022

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 100)
import numpy as np
import os

In [ ]:
sub_folder = 'Nov_18'
year = '2022'
# Export tag is used in output filenames.
export_tag = sub_folder
# Backward-compatible alias for legacy cells.
analysis_name = export_tag
days_count = 17
date_input = '{}-11-01'.format(year) #改日期！！！
date_low_limit = pd.to_datetime('{}-11-01'.format(year)) #改日期！！！
date_high_limit = pd.to_datetime('{}-11-17'.format(year)) #改日期！！！

raw_data_path = str((DATA_ROOT / year / sub_folder).as_posix()) + '/'
outputs_path = str((OUTPUT_ROOT / year / sub_folder).as_posix()) + '/'
mapping_path = str((MAPPING_ROOT).as_posix()) + '/'
final_path = str((OUTPUT_ROOT / 'deliverables').as_posix()) + '/'
process_check_path = str((OUTPUT_ROOT / 'process_check').as_posix()) + '/'

In [ ]:
def format_standardization(df_, KA_, date_mode, date_, target_, branch_, sku_, province_, price_, date_low_, date_high_):
    """标准化 + 映射 + 价格富化（保持原有业务口径）。"""

    df_ = df_.copy()

    if date_mode == 0:
        df_.loc[:, target_[0]] = pd.to_datetime(df_.loc[:, target_[0]]).dt.date
    else:
        df_[target_[0]] = pd.to_datetime(date_)

    if len(target_) > 6:
        df_['Offtake_SKU'] = df_[target_[4]].astype(str) + df_[target_[5]].astype(str)
    else:
        df_['Offtake_SKU'] = df_[target_[4]].astype(str)

    df_['KA'] = KA_
    df_ = df_[[target_[0], target_[1], target_[2], target_[3], 'Offtake_SKU', target_[-1]]]
    df_.columns = ['销售日期', 'KA', 'Offtake_分部', '门店', 'Offtake_SKU', '数量']

    df_ = df_[(df_.销售日期 >= date_low_) & (df_.销售日期 <= date_high_)]

    branch_map = branch_.copy()
    if KA_ != 'DDI新增':
        branch_map = branch_map.loc[branch_map.KA == KA_]
    if 'KA' in branch_map.columns:
        branch_map = branch_map.drop(columns=['KA'])
    branch_map = branch_map.reset_index(drop=True)

    df_ = df_.merge(branch_map, how='left', on='Offtake_分部', suffixes=('', '_Mapped'))
    print('Non Mapped Branch: ', list(set(df_.loc[df_.连锁分部名称.isna(), 'Offtake_分部'])))
    df_.dropna(subset=['连锁分部名称'], inplace=True)

    df_ = df_.merge(province_, how='left', on='城市', suffixes=('', '_Mapped'))
    print('Non Mapped City: ', list(set(df_.loc[df_.省份.isna(), '城市'])))
    df_.loc[df_.省份.isna(), '省份'] = '未知'

    df_ = df_.merge(sku_, how='left', on='Offtake_SKU', suffixes=('', '_Mapped'))
    print('Non Mapped SKU: ', list(set(df_.loc[df_.品规.isna(), 'Offtake_SKU'])))
    df_.dropna(subset=['品规'], inplace=True)
    df_.pop('Offtake_SKU')

    df_ = df_.merge(price_, how='left', on='品规')
    price_col = price_.columns[-1]
    df_['Offtake'] = df_['数量'] * df_[price_col]
    df_.rename(columns={price_col: '考核价'}, inplace=True)

    return df_

In [ ]:
def files_integration(KA_, path_, sheet_name_, skip_row_):
    """同构多文件整合：过滤临时文件并稳定拼接。"""

    integration_files_path = os.path.join(path_, KA_)
    file_list = [
        f for f in os.listdir(integration_files_path)
        if f != 'Archived' and ('$' not in f) and f.lower().endswith(('.xlsx', '.xls'))
    ]

    df_list = []
    for file_name in sorted(file_list):
        file_path = os.path.join(integration_files_path, file_name)
        if sheet_name_ != '':
            df = pd.read_excel(file_path, sheet_name=sheet_name_, skiprows=skip_row_)
        else:
            df = pd.read_excel(file_path, skiprows=skip_row_)
        df_list.append(df)

    if not df_list:
        return pd.DataFrame()

    df_last = pd.concat(df_list, ignore_index=True)
    df_last.dropna(how='all', inplace=True)
    df_last.reset_index(inplace=True, drop=True)
    return df_last

In [ ]:
def files_integration_Tianji(KA_, path_, sheet_name_, skip_row_):
    """天济场景整合：按文件名补充分部信息并拼接。"""

    integration_files_path = os.path.join(path_, KA_)
    file_list = [
        f for f in os.listdir(integration_files_path)
        if ('$' not in f) and f.lower().endswith(('.xlsx', '.xls'))
    ]

    df_list = []
    for file_name in sorted(file_list):
        file_path = os.path.join(integration_files_path, file_name)
        if sheet_name_ != '':
            df = pd.read_excel(file_path, sheet_name=sheet_name_, skiprows=skip_row_)
        else:
            df = pd.read_excel(file_path, skiprows=skip_row_)

        if '益生天济' in file_name:
            df['Offtake_分部'] = '益生天济'
        else:
            df['Offtake_分部'] = '天济大药房'

        df_list.append(df)

    if not df_list:
        return pd.DataFrame()

    df_last = pd.concat(df_list, ignore_index=True)
    df_last.dropna(how='all', inplace=True)
    df_last.reset_index(inplace=True, drop=True)
    return df_last

In [ ]:
def date_expansion_by_day(df_, date_input_, days_count_):
    """将周期数据按日均分展开到日级记录。"""

    df_process = df_.copy()
    df_process['数量'] = df_process['数量'] / days_count_
    df_process['Offtake'] = df_process['Offtake'] / days_count_

    start_date = pd.to_datetime(date_input_)
    expanded_dates = pd.date_range(start=start_date, periods=days_count_, freq='D')

    df_final = pd.concat(
        [df_process.assign(销售日期=current_date) for current_date in expanded_dates],
        ignore_index=True,
    )
    return df_final

In [ ]:
def non_mapped_phaymacy(df_):
    print('Non Mapped Pharmacy:')
    for i in list(set(df_.loc[df_.连锁分部名称.isna(), '门店名称'])):
        print(i)

In [ ]:
df_branch = pd.read_excel(mapping_path + 'Branch_Mapping.xlsx').iloc[:,:10]
df_branch.drop_duplicates(subset=['Offtake_分部'],inplace=True)
df_branch.dropna(subset=['Offtake_分部'],inplace=True)
df_branch.reset_index(inplace=True,drop=True)

In [ ]:
df_SKU = pd.read_excel(mapping_path + 'SKU_Mapping.xlsx',sheet_name='SKU')
df_SKU = df_SKU[df_SKU.Platform == 'Retail']
df_SKU.drop_duplicates(subset=['Offtake_SKU'],inplace=True)
df_SKU.dropna(subset=['Offtake_SKU'],inplace=True)
df_SKU = df_SKU[['Offtake_SKU',"品规","品类","品类2"]]
df_SKU.reset_index(inplace=True,drop=True)

In [ ]:
price_year = '2022' #考核价年份
CAT_columns = ['品规','考核价{}'.format(price_year)]
df_price = pd.read_excel(mapping_path + 'SKU_Mapping.xlsx',sheet_name='CAT')[CAT_columns]
df_price.drop_duplicates(subset=['品规'],inplace=True)
df_price.dropna(subset=['品规'],inplace=True)
df_price.reset_index(inplace=True,drop=True)

In [ ]:
df_province = pd.read_excel(mapping_path + 'Province_City_Region_Mapping.xlsx').iloc[:,:2]
df_province.drop_duplicates(subset=['城市'],inplace=True)
df_province.dropna(subset=['城市'],inplace=True)
df_province.reset_index(inplace=True,drop=True)

### 国大

## 客户：国大

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '国大'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','区域名称','门店名称','商品名称','规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：竹林

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '竹林'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)

df['数量'].sum()

In [ ]:
target_column = ['日期','KA','公司名称','门店名称','商品名称','规格','数量']
KA_name = '国大'
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
#df_process.loc[:,'KA'] = '国大' #销量归属国大
KA_name = '竹林' #for file output
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：一心堂

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '一心堂'
df = files_integration(KA_name,raw_data_path,'销售明细',1)
df['销售数量'].sum()

In [ ]:
target_column = ['日期','KA','公司','门店名称','商品名称','销售数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：漱玉

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '漱玉'
df = files_integration(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','公司名称','门店名称','品名','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：老百姓

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
print (days_count)
KA_name = '老百姓'
#df = files_integration(KA_name,raw_data_path,'',0) #加盟文件改“省公司”
#df['数量'].sum()

In [ ]:
file_list = os.listdir(raw_data_path + KA_name)
file_list

In [ ]:
for file_ in file_list:
    if not('药房' in file_) and not('Archived' in file_):
        raw_file = file_
        print ('Current-file-name:{}'.format(raw_file))
        break
        
df = pd.read_excel(raw_data_path + KA_name + '\\' + raw_file)
#df['门店'] = '老百姓门店'
df['数量'].sum()

In [ ]:
target_column = ['日期','KA','省公司','业务部门','商品名称','规格','数量'] #2022,省公司
df_process = format_standardization(df,KA_name,1,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

In [ ]:
df_process = date_expansion_by_day(df_process,date_input,days_count)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 老百姓健康药房

In [ ]:
for file_ in file_list:
    if ('健康' in file_):
        raw_file = file_
        print ('Current-file-name:{}'.format(raw_file))
        break
        
df = pd.read_excel(raw_data_path + KA_name + '\\' + raw_file)
df['KA'] = '老百姓'
df['数量'].sum()

In [ ]:
target_column = ['销售时间','KA','所属省公司','业务部门','商品名称','规格','数量'] #所属省公司
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_健康药房_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：DDI新增

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = 'DDI新增'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)

df = df[df.连锁总部 == '纯销'].copy()
df.reset_index(drop=True,inplace=True)
df.rename(columns={'品类': '品类_bk'},inplace=True)
df['销量'].sum()

In [ ]:
target_column = ['日期','KA','医药公司','下游名称','产品','销量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.pop('KA')
df_process.pop('KA_Mapped')
df_process = df_process.merge(df_branch[['连锁分部名称','KA']],on='连锁分部名称')
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 海王+健之佳 月交付

## 客户：海王健之佳

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '海王健之佳'
file_name = '{}_{}_{}.xlsx'.format(KA_name,'月交付',analysis_name)
df = pd.read_excel(raw_data_path + file_name)

## 客户：海王

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '海王'
df_process = df[df.连锁总部 == KA_name].copy()
df_process.reset_index(drop=True,inplace=True)
df_process.rename(columns={'品类': '品类_bk'},inplace=True)
df_process['销量'].sum()

In [ ]:
target_column = ['日期','KA','连锁出货','下游名称','产品','销量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.pop('KA')
#df_process.pop('KA_Mapped')
df_process = df_process.merge(df_branch[['连锁分部名称','KA']],on='连锁分部名称')
df_process.to_csv(outputs_path + '{}_月交付_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：高济

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '高济'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)

df['销售数量'].sum()

In [ ]:
target_column = ['业务日期','KA','企业名称','门店名称','商品名称','规格','销售数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：先声

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '先声'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)

KA_name = '高济'
df['KA'] = KA_name
df['数量'].sum()

In [ ]:
df_process = df.copy()
for i in range(df_process.shape[0]):
    pharmacy_ = df_process.loc[i,'客户名称']
    
    if '徐州' in pharmacy_:
        df_process.loc[i,'分部'] = '先声再康（徐州）连锁药店有限公司'
    else:
        df_process.loc[i,'分部'] = '先声再康江苏药业有限公司'

In [ ]:
target_column = ['开单日期','KA','分部','客户名称','品名','规格','数量']

df_final = format_standardization(df_process,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)

KA_name = '先声'
df_final.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_final['数量'].sum()

### 海王

In [ ]:
file_name = '{}_{}.xlsx'.format('海王健之佳',analysis_name) #海王健之佳

df = pd.read_excel(raw_data_path + file_name,sheet_name='Sell out Details')
df['quantity'].sum()

In [ ]:
KA_name = '海王'
target_column = ['Sales Date','KA','ChainShipment','Buyer Name','Product Name','quantity']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：健之佳

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '健之佳'
target_column = ['Sales Date','KA','ChainShipment','Buyer Name','Product Name','quantity']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

In [ ]:
KA_name = '海王'
df = files_integration(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
df_pharmacy_mapping = pd.read_excel(mapping_path + '海王_Pharmacy_Mapping.xlsx')
df_pharmacy_mapping.head()

In [ ]:
target_column = ['日期','KA','销售方代码','采购方名称','产品名称','产品规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process['数量'].sum()

In [ ]:
df_final = df_process.merge(df_pharmacy_mapping[['医药公司代码','下游名称','下游标准名称']], how='left', left_on=['Offtake_分部','门店'],right_on=['医药公司代码','下游名称'])
df_final.head(2)

In [ ]:
df_final.loc[pd.isna(df_final.下游标准名称),['门店','连锁分部名称']].drop_duplicates().reset_index(drop=True)

In [ ]:
df_final.loc[pd.isna(df_final.下游标准名称),'下游标准名称'] = \
df_final.loc[pd.isna(df_final.下游标准名称),'连锁分部名称'] + df_final.loc[pd.isna(df_final.下游标准名称),'门店']

In [ ]:
df_final.pop('门店')
df_final.pop('下游名称')
df_final.pop('医药公司代码')
df_final.rename(columns={'下游标准名称': '门店'},inplace=True)
df_final['数量'].sum()

In [ ]:
df_final['Offtake'].sum()

In [ ]:
df_final.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)

### 海王原始文件 - 倍通

In [ ]:
KA_name = '海王'
df = files_integration(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
df.loc[(df.销售方代码=='K000153') & (df.采购方代码.str.contains('V')),'销售方代码'] = 'K000322'

In [ ]:
target_column = ['日期','KA','销售方代码','采购方名称','产品名称','产品规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process['数量'].sum()

In [ ]:
df_pharmacy_mapping = pd.read_excel(mapping_path + '海王_Pharmacy_Mapping.xlsx')

In [ ]:
df_final = df_process.merge(df_pharmacy_mapping[['医药公司代码','下游名称','下游标准名称']], how='left', left_on=['Offtake_分部','门店'],right_on=['医药公司代码','下游名称'])

In [ ]:
df_final.loc[pd.isna(df_final.下游标准名称),['门店','连锁分部名称']].drop_duplicates().reset_index(drop=True)

In [ ]:
#df_final.loc[pd.isna(df_final.下游标准名称),'下游标准名称'] = \
#df_final.loc[pd.isna(df_final.下游标准名称),'连锁分部名称'] + df_final.loc[pd.isna(df_final.下游标准名称),'门店']

In [ ]:
df_final = df_final.iloc[:,:-3]

In [ ]:
df_final.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)

### 健之佳原文件

In [ ]:
KA_name = '健之佳'
df = files_integration(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
df_process = df.copy()
df_process.head(2)

In [ ]:
df_process.loc[df_process.销售方代码 == 'C002958','销售方代码'] = 'K000665'
df_process.loc[df_process.销售方代码 == 'C003131','销售方代码'] = 'K000667'

In [ ]:
target_column = ['日期','KA','销售方代码','采购方名称','产品名称','产品规格','数量']
df_process = format_standardization(df_process,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process['数量'].sum()

In [ ]:
df_pharmacy_mapping = pd.read_excel(mapping_path + '健之佳_Pharmacy_Mapping.xlsx')
df_pharmacy_mapping.loc[df_pharmacy_mapping.医药公司代码 == 'C002958','医药公司代码'] = 'K000665'
df_pharmacy_mapping.loc[df_pharmacy_mapping.医药公司代码 == 'C003131','医药公司代码'] = 'K000667'
df_pharmacy_mapping.head()

In [ ]:
df_final = df_process.merge(df_pharmacy_mapping[['医药公司代码','下游名称','下游标准名称']], how='left', left_on=['Offtake_分部','门店'],right_on=['医药公司代码','下游名称'])

In [ ]:
df_final.loc[pd.isna(df_final.下游标准名称),['门店','连锁分部名称']].drop_duplicates().reset_index(drop=True)

In [ ]:
df_final.loc[pd.isna(df_final.下游标准名称),'下游标准名称'] = \
df_final.loc[pd.isna(df_final.下游标准名称),'连锁分部名称'] + df_final.loc[pd.isna(df_final.下游标准名称),'门店']

In [ ]:
df_final.pop('门店')
df_final.pop('下游名称')
df_final.pop('医药公司代码')
df_final.rename(columns={'下游标准名称': '门店'},inplace=True)
df_final['数量'].sum()

In [ ]:
df_final.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)

## 客户：大参林

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '大参林'
#file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = files_integration(KA_name,raw_data_path,'',0)
#df = pd.read_excel(raw_data_path + file_name)
df['数量'].sum()

In [ ]:
# Pharmacy name revising - 分店 --> 店

pharmacy_column = '门店描述'

for i in range(df.shape[0]):
    pharmacy_ = df.loc[i,pharmacy_column]
    if '分店' in pharmacy_:
        print ('Please check 分店 in Process folder')
        break

In [ ]:
target_column = ['销售日期','KA','营运区','门店描述','商品名称','商品规格','数量'] #门店描述
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 大参林 - New Version

In [ ]:
KA_name = '大参林'
#file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = files_integration(KA_name,raw_data_path,'',0)
#df = pd.read_excel(raw_data_path + file_name)
df['数量'].sum()
df.drop(['营运区'],axis = 1)
print ('Done')

In [ ]:
df_pharmacy_mapping = pd.read_excel(mapping_path + '大参林_Pharamcy_Mapping.xlsx')[['门店描述','终端对应连锁名称']]
df_pharmacy_mapping.drop_duplicates(subset=['门店描述'],inplace=True)
df = df.merge(df_pharmacy_mapping, how='left', on='门店描述').reset_index(drop=True)

In [ ]:
target_column = ['销售日期','KA','终端对应连锁名称','门店描述','商品名称','商品规格','数量'] #门店描述
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 安徽国胜

## 客户：安徽国胜

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '安徽国胜'
df = files_integration(KA_name,raw_data_path,'',0)
#df = pd.read_excel(raw_data_path + file_name)
df['数量'].sum()

In [ ]:
#KA_name = '安徽国胜'
#file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
#df = pd.read_excel(raw_data_path + file_name,sheet_name='{}'.format('商品信息0'))
df['分部'] = ''
for n in range(df.shape[0]):
    df.loc[n,'分部'] = df.loc[n,'销售方名称'].split('国胜大药房')[0] 
df['数量'].sum()

In [ ]:
target_column = ['日期','KA','分部','销售方名称','产品名称','产品规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：益丰

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '益丰'
df = files_integration(KA_name,raw_data_path,'',1)
df_pharmacy_mapping = pd.read_excel(mapping_path + '益丰_Pharmacy_Mapping.xlsx')
df_pharmacy_mapping.drop_duplicates(subset=['门店名称'],inplace=True)
df = df.merge(df_pharmacy_mapping, how='left', on='门店名称').reset_index(drop=True)
non_mapped_phaymacy(df) #加公司名
df['销售数量'].sum()

In [ ]:
df.loc[df.连锁分部名称.isna()][['门店名称','公司名']].reset_index(drop=True).drop_duplicates(subset=['门店名称']).to_excel(process_check_path + 'Yifeng.xlsx')

In [ ]:
target_column = ['销售日期','KA','连锁分部名称','门店名称','商品名','销售数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：西安怡康

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '西安怡康'
df = files_integration(KA_name,raw_data_path,'',0)
print (df['数量'].sum())

In [ ]:
#KA_name = '西安怡康'
#file_name = '{}_{}.xls'.format(KA_name,analysis_name)
#df = pd.read_excel(raw_data_path + file_name)
#df.head()
#print (df['数量'].sum())

df['分部'] = df['门店名称']

for row_ in range(df.shape[0]):
    pharmacy_ = df.loc[row_,'分部']
        
    if ("（" in pharmacy_):
        df.loc[row_,'分部'] = pharmacy_.split('公司')[0] + '公司）'
        
    elif ("(" in pharmacy_):
        df.loc[row_,'分部'] = pharmacy_.split('公司')[0] + '公司)'
    
    else: 
        df.loc[row_,'分部'] = pharmacy_.split('公司')[0] + '公司'
        
df['日期'] = df['日期'].str.split(' ',expand=True).loc[:,0]
df['日期'] = pd.to_datetime(df['日期'])

In [ ]:
target_column = ['日期','KA','分部','门店名称','货品名称','规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：吉林大药房

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '吉林大药房'
df = files_integration(KA_name,raw_data_path,'',1)
df['数量'].sum()

In [ ]:
target_column = ['发生时间','KA','分部','业务机构','药品名称','药品规格','数量']
df['分部'] = KA_name
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：康佰家

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '康佰家'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)

df = pd.read_excel(raw_data_path + file_name)
print (df['数量'].sum())

In [ ]:
target_column = ['销售日期','KA','分部','机构名称','商品名称','规格','数量']
df['分部'] = KA_name
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 鹭燕

## 客户：鹭燕

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '鹭燕'
df = files_integration(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','公司名称','门店名','货品名称','规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：天济

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '天济'
df = files_integration_Tianji(KA_name,raw_data_path,'',0)
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','Offtake_分部','门店名称','商品名称','商品规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：济南平嘉

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '济南平嘉'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)

df = pd.read_excel(raw_data_path + file_name,skiprows=1)
df['数量'].sum()

In [ ]:
target_column = ['业务日期','KA','分部','门店','货品名称','规格','数量']
df['分部'] = KA_name
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：汇丰医药

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '汇丰医药'
df = files_integration(KA_name,raw_data_path,'',0)
df['销售数据'].sum()

In [ ]:
df_pharmacy_mapping = pd.read_excel(mapping_path + '汇丰医药_Pharmacy_Mapping.xlsx')
df_pharmacy_mapping.drop_duplicates(subset=['门店名称'],inplace=True)
df = df.merge(df_pharmacy_mapping, how='left', on='门店名称').reset_index(drop=True)
non_mapped_phaymacy(df)

In [ ]:
target_column = ['销售日期','KA','连锁分部名称','门店名称','商品名称','商品规格','销售数据']
df['分部'] = KA_name
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：贵州一品

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '贵州一品'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)
df = df[df.序号 != '总计']
df['销售日期'] = df['销售日期'].str.split(' ',expand=True).loc[:,0]
df['销售日期'] = pd.to_datetime(df['销售日期']).dt.date
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','分部','门店名称','品名','规格','数量']
df['分部'] = KA_name
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：河北神威

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
days_count
KA_name = '河北神威'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)

df['业务机构_匹配'] = ''
for i in range(df.shape[0]):
    branch_ = df.loc[i,'门店名称']
    if '唐山' in branch_:
        df.loc[i,'业务机构_匹配'] = '神威大药房连锁（唐山）有限公司'
    else:
        df.loc[i,'业务机构_匹配'] = '河北神威大药房连锁有限公司'
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','业务机构_匹配','门店名称','品名','规格','数量']
df_process = format_standardization(df,KA_name,1,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

In [ ]:
df_process = date_expansion_by_day(df_process,date_input,days_count)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：山东燕喜堂

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '山东燕喜堂'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)
df['业务机构_匹配'] = KA_name
df.rename(columns={'数量 ': '数量'},inplace=True)
print (df['数量'].sum())

In [ ]:
target_column = ['销售日期','KA','业务机构_匹配','机构名称','商品名称','规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### 泉源堂

## 客户：泉源堂

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '泉源堂'
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name,skiprows=1)
df = df.loc[df.序号!='序号',:]
df = df.reset_index(drop=True)

df['门店编号'] = df['门店编号'].str[:4] 
for i in list(df.columns):
    if '销售数量' in i:
        volume_ = i
df.rename(columns={volume_: '数量'},inplace=True)
df['数量'].sum()

In [ ]:
target_column = ['销售日期','KA','门店编号','门店名称','商品名称','规格','数量']
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

## 客户：张仲景

- 本节处理该客户数据的标准化与映射。
- 运行前请确认参数区已正确配置。

In [ ]:
KA_name = '张仲景'
print (days_count)
file_name = '{}_{}.xlsx'.format(KA_name,analysis_name)
df = pd.read_excel(raw_data_path + file_name)
df['数量'].sum()

In [ ]:
target_column = ['日期','KA','分部','门店名称','品名','规格','数量'] #门店名称
df['分部'] = KA_name

#for website data
#df_process = format_standardization(df,KA_name,1,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)

# for manual data
df_process = format_standardization(df,KA_name,0,date_input,target_column,df_branch,df_SKU,df_province,df_price,date_low_limit,date_high_limit)

df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

In [ ]:
#df_process = date_expansion_by_day(df_process,date_input,days_count)
df_process.to_csv(outputs_path + '{}_{}.csv'.format(KA_name,analysis_name),encoding='utf-8_sig',index=False)
df_process['数量'].sum()

### Integration

In [ ]:
import os

In [ ]:
file_list = os.listdir(outputs_path)

try:
    file_list.remove('Archived')
except:
    pass

print (len(file_list))
file_list

file_list_non_duplicated = []
for i in file_list:
    ka_ = i.split('_')[0]
    file_list_non_duplicated.append(ka_)
    
print ('Non-duplicated KA amount = {}'.format(len(list(set(file_list_non_duplicated)))))

In [ ]:
df_all = []
for file in file_list:
    df_current = pd.read_csv(outputs_path + file)
    df_current['销售日期'] = pd.to_datetime(df_current['销售日期'])
    df_all.append(df_current)

if len(df_all) == 0:
    raise ValueError('No integrated files found in outputs_path.')

df_last = pd.concat(df_all, ignore_index=True)
df_last.to_csv(final_path + '{}_{}_{}.csv'.format('Offtake_Integration',analysis_name,year), index=False, encoding='utf-8_sig')

### Date range generation

In [ ]:
from datetime import datetime

In [ ]:
current_date = datetime.today().strftime('%Y-%m-%d')

date_list = pd.date_range(start = '1/1/2017', end = current_date)

df_date = pd.DataFrame(columns = ['销售日期'])
df_date['销售日期'] = date_list


outputs_path = str((MAPPING_ROOT).as_posix()) + '/'
df_date.iloc[:-1,:].to_csv(outputs_path + 'Date_Range.csv',index=False,encoding='utf-8_sig')
df_date.iloc[:-1,:].tail()

### End